# 9장 실습 ③ — ImageNet 사전학습 모델

**PyTorch 판**

실무에서는 직접 원천 과제를 학습시키지 않습니다.
**이미 잘 학습된 것을 받아 씁니다.**

> ⚠ 가중치 내려받기가 막힌 환경에서는 건너뜁니다.
> Kaggle이나 Colab에서 돌리시면 받아집니다.

## 9.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot
from dlbook.data import Split

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 9.1 원천 과제와 목표 과제

**시험 800개·검증 400개를 고정**하고 학습 데이터만 줄입니다.
초고에서는 통째로 줄였다가 시험 12개짜리 표를 만들 뻔했습니다. (7장 §7.6)

In [ ]:
# 원천 과제 — 원·사각형·삼각형 6,000개
xs, ys = data.shapes(6000, seed=42, classes=(0, 1, 2))
src = data.split(xs, ys, val_ratio=0.15, test_ratio=0.15, seed=42)
print("원천 과제:", src)

# 목표 과제 — 십자·마름모. **원천 과제에 없던 도형이다.**
xt, yt = data.shapes(3000, seed=7, classes=(3, 4))

# ★ 시험 800개·검증 400개를 **고정**하고 학습 데이터만 줄인다.
#   (7장 §7.6 — 시험 데이터가 작으면 숫자를 믿을 수 없다.
#    초고에서 통째로 줄였다가 시험 12개짜리 표를 만들 뻔했다.)
x_test, y_test = xt[:800], yt[:800]
x_val, y_val = xt[800:1200], yt[800:1200]
pool_x, pool_y = xt[1200:], yt[1200:]

def target(n):
    """학습 데이터 n개짜리 목표 과제. 시험·검증은 언제나 같다."""
    return Split(pool_x[:n], pool_y[:n], x_val, y_val, x_test, y_test)

fig = plot.image_grid(np.concatenate([xs[:8], xt[:8]]),
                      np.concatenate([ys[:8], yt[:8] + 3]), n=16, cols=8,
                      class_names=list(data.SHAPE_CLASSES))
plt.show()
print("위: 원천 과제의 도형 / 아래: 목표 과제의 도형")

## 9.2 학습 함수 — 여기만 판마다 다릅니다

`train_transfer()` 안의 **①얼린다 → ②분류부만 → ③푼다 → ④낮은 학습률**
순서를 눈여겨보십시오. **처음부터 풀면 배운 것이 망가집니다.**

In [ ]:
import copy

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
SRC_STATE = {}

def to_nchw(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32).permute(0, 3, 1, 2)

def _cnn(n_classes, seed=42):
    dlbook.set_seed(seed)
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
        nn.Linear(64, n_classes),
    ).to(device)

def _fit(model, sp, epochs, lr):
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    dl = DataLoader(TensorDataset(to_nchw(sp.x_train),
                                  torch.tensor(sp.y_train, dtype=torch.long)),
                    batch_size=32, shuffle=True)
    for _ in range(dlbook.smoke.epochs(epochs)):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(to_nchw(sp.x_test).to(device)).argmax(1).cpu().numpy()
    return metrics.accuracy(sp.y_test, pred)

def train_source(sp):
    """원천 과제를 학습하고 가중치를 남긴다."""
    m = _cnn(3)
    acc = _fit(m, sp, 15, 0.001)
    SRC_STATE.update(copy.deepcopy(m.state_dict()))
    return acc

def train_scratch(sp):
    """목표 과제를 처음부터 학습한다."""
    return _fit(_cnn(2), sp, 30, 0.001)

def train_transfer(sp, finetune=False):
    """가져온 특징 추출부로 목표 과제를 학습한다. 얼렸다가 푸는 순서를 지킨다."""
    src_model = _cnn(3)
    src_model.load_state_dict(SRC_STATE)
    base = nn.Sequential(*list(src_model.children())[:7])   # Flatten 까지
    for p in base.parameters():
        p.requires_grad = False                             # ① 얼린다
    dlbook.set_seed(42)
    model = nn.Sequential(base, nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
                          nn.Linear(64, 2)).to(device)
    acc = _fit(model, sp, 30, 0.001)                        # ② 분류부만
    if not finetune:
        return acc
    for p in base.parameters():
        p.requires_grad = True                              # ③ 푼다
    return _fit(model, sp, 15, 0.0001)                      # ④ 낮은 학습률로

def train_pretrained(sp):
    """ImageNet 사전학습 모델 (torchvision)."""
    import torch.nn.functional as F
    from torchvision.models import MobileNet_V2_Weights, mobilenet_v2
    w = MobileNet_V2_Weights.IMAGENET1K_V1
    base = mobilenet_v2(weights=w)
    for p in base.parameters():
        p.requires_grad = False
    base.classifier = nn.Sequential(nn.Linear(1280, 64), nn.ReLU(), nn.Linear(64, 2))
    base = base.to(device)

    class Wrap(nn.Module):
        def __init__(self, net):
            super().__init__(); self.net = net
        def forward(self, x):
            x = F.interpolate(x, size=(96, 96), mode="bilinear")
            x = x.repeat(1, 3, 1, 1) * 2.0 - 1.0
            return self.net(x)
    return _fit(Wrap(base).to(device), sp, 15, 0.001)

## 9.3 사전학습 모델 가져오기

In [ ]:
# ImageNet 사전학습 모델. 내려받기가 막힌 환경에서는 건너뛴다.
try:
    acc = train_pretrained(target(400))
    dlbook.record("ch09_pretrained_acc", acc)
    print(f"사전학습 모델 전이학습 시험 정확도 {acc:.3f}")
except Exception as e:
    print(f"⚠ 사전학습 가중치를 받지 못했습니다 ({type(e).__name__}).")
    print("  Kaggle이나 Colab에서 돌리시면 받아집니다.")
    print("  §9.3의 도형 실험만으로도 전이학습의 논리는 확인됩니다.")

## 정리

- 실무의 전이학습은 **ImageNet으로 학습된 모델**을 받아 쓰는 것입니다.
- **전처리를 모델에 맞춰야 합니다.** `/255.0` 만 하고 넘기면 안 됩니다.
- 입력 크기와 채널 수도 맞춰야 합니다.
- **ImageNet 정확도가 높다고 내 문제에서도 최고인 것은 아닙니다.**
  두세 개를 골라 **검증 데이터로** 비교하십시오. (7장 §7.6)

### 연습

1. MobileNetV2 대신 ResNet50, VGG16으로 바꿔 비교하십시오.
2. `preprocess_input` 을 **빼고** 돌리면 성능이 얼마나 떨어집니까.
3. 이 결과로 본문 §9.4의 빈 표를 채우십시오.